# Module 04.05 — Maps (`map`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.5 Maps (`map`)

The Maps application stores geospatial visualizations as `map` saved objects. A map can
contain multiple **layers** — Documents (scatter of ES documents), Heat maps, Clusters,
EMS boundaries (country/region outlines from Elastic Maps Service), and Grid aggregations.
Each layer has its own data source configuration, style rules, and visibility filters.

The schema stores layers as `layerListJSON` (another JSON-in-JSON pattern), with each
entry carrying a `sourceDescriptor` that describes where the geo data comes from.
Document layers reference a Data View and a geo field; EMS layers reference an external
tile manifest. This means a map with EMS layers has no `references` entries at all —
it is self-contained — while a Document layer produces a reference to the Data View.

For restore purposes, maps behave like Lens charts: they are fully captured in the
`kibana` feature state, they restore cleanly as long as the referenced Data Views exist,
and they migrate well between patch versions. They do *not* snapshot the underlying
geo index data — that is captured separately as a normal index snapshot.

### Create in Kibana UI
1. Go to **Maps → Create map**
2. Click **Add layer → Documents**
3. Select the eCommerce data view and the `geoip.location` field
4. Save as `Training — Customer Locations`

In [ ]:
heading('4.5 Maps — inspect saved objects')

maps = find_saved_objects('map')
console.print(f'  Found {len(maps)} map(s)')
if maps:
    m = maps[0]
    # Maps can be large; show structure only
    attrs = m['attributes']
    console.print(f"  Title: {attrs.get('title')}")
    try:
        map_state = json.loads(attrs.get('mapStateJSON', '{}'))
        pp(map_state, 'mapStateJSON')
    except Exception:
        pass
    info('Key fields:')
    info('  layerListJSON   — array of layer descriptors (type, source, style)')
    info('  mapStateJSON    — zoom, center coordinates, timeFilters')
    info('  uiStateJSON     — layer visibility state')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/maps', 'Maps — view and edit map saved objects')

In [ ]:
heading('4.5 Maps — create via API')

ecom_dv = next((o for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', '')), None)
if not ecom_dv:
    raise RuntimeError('eCommerce data view not found — re-run 00_setup')
ecom_dv_id = ecom_dv['id']

existing_map = next((o for o in find_saved_objects('map') if o['attributes'].get('title') == 'Training — Customer Locations'), None)
if existing_map:
    map_id = existing_map['id']
    info(f'Map already exists: id={map_id}')
else:
    map_resp = kibana_post('/api/saved_objects/map', {
        'attributes': {
            'title': 'Training — Customer Locations',
            'description': '',
            'mapStateJSON': json.dumps({
                'zoom': 3, 'lon': 0, 'lat': 40,
                'timeFilters': {'from': 'now-30d', 'to': 'now'},
                'query': {'language': 'kuery', 'query': ''},
                'filters': [], 'refreshConfig': {'isPaused': True, 'interval': 0},
            }),
            'layerListJSON': json.dumps([{
                'id': 'ems-boundaries',
                'label': 'World Countries',
                'minZoom': 0, 'maxZoom': 24, 'alpha': 0.5,
                'visible': True, 'style': {'type': 'VECTOR'},
                'type': 'EMS_VECTOR_TILE',
                'sourceDescriptor': {'type': 'EMS_TMS', 'isAutoSelect': True},
            }, {
                'id': 'doc-layer',
                'label': 'Customer geo',
                'minZoom': 0, 'maxZoom': 24, 'alpha': 0.75,
                'visible': True,
                'style': {'type': 'VECTOR', 'properties': {}},
                'type': 'GEOJSON_VECTOR',
                'sourceDescriptor': {
                    'type': 'ES_SEARCH',
                    'id': 'doc-source',
                    'indexPatternId': ecom_dv_id,
                    'indexPatternRefName': 'layer_0_source_index_pattern',
                    'geoField': 'geoip.location',
                    'filterByMapBounds': True,
                    'topHitsSplitField': '',
                    'topHitsSize': 1,
                    'tooltipProperties': ['customer_full_name', 'taxful_total_price'],
                    'applyGlobalQuery': True, 'applyGlobalTime': True,
                },
            }]),
            'uiStateJSON': json.dumps({'isLayerTOCOpen': True, 'openTOCDetails': []}),
        },
        'references': [{'id': ecom_dv_id, 'name': 'layer_0_source_index_pattern', 'type': 'index-pattern'}],
    })
    map_id = map_resp['id']
    success(f'Map created: id={map_id}')

obj = kibana_get(f'/api/saved_objects/map/{map_id}')
info('Key fields:')
info('  mapStateJSON   — viewport: zoom, center, time filter')
info('  layerListJSON  — array of layer configs (type, source, style)')
info('  references     — data view IDs used by document layers')
kibana_link(f'/app/maps/map/{map_id}', 'Open Training — Customer Locations in Maps')

In [ ]:
# Ensure map exists before snapshotting (re-runnable)
if not any(o['id'] == map_id for o in find_saved_objects('map')):
    warn('Map missing — recreating')
    ecom_dv_id = next(o['id'] for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', ''))
    map_resp = kibana_post('/api/saved_objects/map', {
        'attributes': {
            'title': 'Training — Customer Locations', 'description': '',
            'mapStateJSON': json.dumps({'zoom': 3, 'lon': 0, 'lat': 40,
                'timeFilters': {'from': 'now-30d', 'to': 'now'},
                'query': {'language': 'kuery', 'query': ''}, 'filters': [],
                'refreshConfig': {'isPaused': True, 'interval': 0}}),
            'layerListJSON': json.dumps([{'id': 'ems-boundaries', 'label': 'World Countries',
                'minZoom': 0, 'maxZoom': 24, 'alpha': 0.5, 'visible': True,
                'style': {'type': 'VECTOR'}, 'type': 'EMS_VECTOR_TILE',
                'sourceDescriptor': {'type': 'EMS_TMS', 'isAutoSelect': True}}]),
            'uiStateJSON': json.dumps({'isLayerTOCOpen': True, 'openTOCDetails': []}),
        },
        'references': [],
    })
    map_id = map_resp['id']

snap_delete_restore_cycle('snap-map-test', 'Map')

kibana_delete(f'/api/saved_objects/map/{map_id}')
warn(f'Accidentally deleted map {map_id}')
assert not any(o['id'] == map_id for o in find_saved_objects('map') if o['attributes'].get('title') == 'Training — Customer Locations'), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-map-test')
time.sleep(3)

restored = find_saved_objects('map')
assert any(o['id'] == map_id for o in restored), 'Map should be restored'
success(f'Map {map_id} successfully restored!')
kibana_link(f'/app/maps/map/{map_id}', 'Verify restored map')